In [1]:
# Imports
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Carregar dados do banco
load_dotenv()

server = os.getenv('SQL_SERVER')
database = os.getenv('SQL_DATABASE')

connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

# Buscar dados combinando as tabelas clientes, solicitacoes e decisoes
# Filtra apenas creditos aprovados para treinar o modelo
query = """
    SELECT 
        clientes.idade,
        clientes.renda_mensal,
        clientes.score_credito,
        clientes.tem_imovel,
        clientes.tem_veiculo,
        clientes.tempo_emprego_anos,
        clientes.qtd_emprestimos_ativos,
        clientes.historico_inadimplencia,
        clientes.possui_restricao,
        clientes.historico_produto,
        solicitacoes_credito.valor_solicitado,
        solicitacoes_credito.prazo_meses,
        solicitacoes_credito.tipo_credito,
        solicitacoes_credito.nivel_risco,
        decisoes.inadimplente
    FROM solicitacoes_credito
    JOIN clientes ON solicitacoes_credito.id_cliente = clientes.id_cliente
    JOIN decisoes ON solicitacoes_credito.id_solicitacao = decisoes.id_solicitacao
    WHERE decisoes.resultado = 'aprovado'
"""

df = pd.read_sql(query, engine)

print(f"Total de registros carregados: {len(df)}")
print(df.head())

Total de registros carregados: 846
   idade  renda_mensal  score_credito  tem_imovel  tem_veiculo  \
0     25       9572.52            440        True         True   
1     38      28171.02            488       False         True   
2     38      20901.54            676       False         True   
3     73       1685.21            828       False         True   
4     60       4452.22            954       False         True   

   tempo_emprego_anos  qtd_emprestimos_ativos  historico_inadimplencia  \
0                 1.4                       3                    False   
1                11.5                       3                    False   
2                29.9                       2                    False   
3                10.2                       1                    False   
4                 3.5                       0                    False   

   possui_restricao   historico_produto  valor_solicitado  prazo_meses  \
0             False  emprestimo_pessoal          